# A2A KG-Guided Activation Steering — Unified Analysis Notebook

This notebook consolidates all model variants and adds:
- **Steering Layer Selection Analysis** — which layer to use and why
- **Steering Activation Fidelity (SAF)** — how faithfully the steer vector is applied
- **Hallucination Severity Index (HSI)** — entity leakage / fabrication rate
- **Tone Consistency Index (TCI)** — style-direction alignment across outputs
- Full pipeline tests for each model variant

---

### Model Reference Table

| Model | `STEER_LAYER` | `STEER_ALPHA` | Hidden dim | Total layers | Depth % | VRAM |
|---|---|---|---|---|---|---|
| `Llama-2-7b-hf` | **20** | 15.0 | 4096 | 32 | 62.5% | ~14 GB |
| `Llama-2-7b-chat-hf` | **20** | 15.0 | 4096 | 32 | 62.5% | ~14 GB |
| `Llama-2-13b-chat-hf` | **28** | 12.0 | 5120 | 40 | 70.0% | ~26 GB |
| `Llama-3.1-8B-Instruct` | **20** | 13.0 | 4096 | 32 | 62.5% | ~16 GB |
| `Llama-3.2-3B-Instruct` | **16** | 12.0 | 3072 | 28 | 57.1% | ~6 GB |
| `Llama-3.2-1B-Instruct` | **10** | 10.0 | 2048 | 16 | 62.5% | ~2 GB |

> **Key insight**: Steering layers are always chosen at ~60–70% depth — the "middle-late" representation zone where syntactic processing is done but semantic/stylistic direction is still being formed.

### Run Order
**Cell 0a → Cell 0b → Cell 1 → Cell 2 → Cell 3 → Cell 4 (select model) → Cell 5 → Cell 6 → Cell 7 → Cell 8 → Cell 9 (layer analysis) → Cell 10–12 (vectors) → Cell 13 (metrics) → Cell 14–16 (tests) → Cell 17–18 (pipeline)**

In [1]:
# ─── Cell 0a — AGGRESSIVE GPU CLEANUP (RUN THIS FIRST EVERY SESSION) ─────────

import gc, sys

if 'main_module' in dir():
    try:
        main_module._model_cache.clear()
        print('  ✓ main_module._model_cache.clear() — done')
    except Exception as e:
        print(f'  cache clear skipped: {e}')

_gone = []
for _v in [k for k in list(globals().keys())
           if any(x in k.lower() for x in ['model', 'tokenizer', 'main_module'])]:
    try:
        del globals()[_v]
        _gone.append(_v)
    except Exception:
        pass
if _gone:
    print(f'  ✓ Deleted globals: {_gone}')

_unloaded = [k for k in list(sys.modules.keys()) if 'main_module' in k]
for _m in _unloaded:
    del sys.modules[_m]
if _unloaded:
    print(f'  ✓ Unloaded from sys.modules: {_unloaded}')

gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved  = torch.cuda.memory_reserved()  / 1e9
        total     = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'\n  GPU         : {torch.cuda.get_device_name(0)}')
        print(f'  Total VRAM  : {total:.1f} GB')
        print(f'  Allocated   : {allocated:.2f} GB  ← should be ~0.0 GB')
        print(f'  Reserved    : {reserved:.2f} GB')
        print(f'  Approx free : {total - reserved:.1f} GB')
        if allocated > 1.0:
            print('  ⚠  WARNING: VRAM still in use — Runtime → Restart runtime → re-run from Cell 0a.')
        else:
            print('  ✓ VRAM clean — safe to load model.')
    else:
        print('  ⚠  No GPU detected — some models require GPU.')
except ImportError:
    print('  torch not installed yet — run Cell 1 first')


  GPU         : NVIDIA RTX 6000 Ada Generation
  Total VRAM  : 50.9 GB
  Allocated   : 0.00 GB  ← should be ~0.0 GB
  Reserved    : 0.00 GB
  Approx free : 50.9 GB
  ✓ VRAM clean — safe to load model.


In [2]:
# ─── Cell 0b — MOUNT GOOGLE DRIVE FOR PERSISTENT MODEL CACHE ─────────────────

import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CACHE = '/content/drive/MyDrive/hf_model_cache'
    os.makedirs(DRIVE_CACHE, exist_ok=True)
    os.environ['HF_HOME']               = DRIVE_CACHE
    os.environ['TRANSFORMERS_CACHE']    = DRIVE_CACHE
    os.environ['HF_DATASETS_CACHE']     = DRIVE_CACHE
    os.environ['HUGGINGFACE_HUB_CACHE'] = DRIVE_CACHE
    import glob
    existing = (glob.glob(f'{DRIVE_CACHE}/**/*.safetensors', recursive=True)
                + glob.glob(f'{DRIVE_CACHE}/**/*.bin', recursive=True))
    print(f'  ✓ HuggingFace cache → {DRIVE_CACHE}')
    if existing:
        total_gb = sum(os.path.getsize(f) for f in existing) / 1e9
        print(f'  ✓ Found {len(existing)} cached weight file(s) ({total_gb:.1f} GB) — NO re-download needed!')
    else:
        print('  ℹ  No cached weights yet — first run will download to Drive.')
except Exception as e:
    print(f'  Drive mount skipped ({e}) — using /root/.cache (re-downloads every session).')

  Drive mount skipped (No module named 'google.colab') — using /root/.cache (re-downloads every session).


In [3]:
# ─── Cell 1 — Install all dependencies ───────────────────────────────────────
# transformers >= 4.45.0 required for Llama-3.2; 4.43.0+ for Llama-3.1

%pip install -q "transformers>=4.45.0" "pydantic>=2.9.0"

%pip install -q \
    fastapi==0.115.0 \
    uvicorn==0.30.6 \
    httpx==0.27.2 \
    python-dotenv>=1.0.0 \
    langchain-groq>=0.1.9 \
    langchain-core>=0.2.0 \
    rich>=13.7.0 \
    accelerate>=0.34.0 \
    huggingface_hub>=0.24.0 \
    scikit-learn>=1.4.0 \
    numpy>=1.26.0 \
    spacy \
    matplotlib \
    seaborn

try:
    import torch
    print(f"✓ torch already available: {torch.__version__}")
except ImportError:
    !pip install -q "torch>=2.2.0"
    import torch

!python -m spacy download en_core_web_sm

import importlib, transformers
importlib.reload(transformers)
print(f"✓ transformers active version: {transformers.__version__}")
print("✓ All dependencies installed — restart kernel once, then re-run from Cell 1")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
✓ torch already available: 2.10.0+cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 18.2 MB/s  0:00:00m0:00:010:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
✓ transformers active version: 5.9.0
✓ All dependencies installed — restart kernel once, then re-run from Cell 1


In [ ]:
# ─── Cell 2 — Set secrets and authenticate ───────────────────────────────────

import os
from huggingface_hub import login

os.environ["HF_TOKEN"]     = "HF_TOKEN"   # https://huggingface.co/settings/tokens
os.environ["GROQ_API_KEY"] = "GROQ_API_KEY"    # https://console.groq.com

HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN or HF_TOKEN == "hf_YOUR_TOKEN_HERE":
    raise EnvironmentError("HF_TOKEN not set. Replace the placeholder above.")

login(token=HF_TOKEN)
print("✓ Hugging Face login successful")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if GROQ_API_KEY and GROQ_API_KEY != "gsk_YOUR_KEY_HERE":
    print("✓ GROQ_API_KEY found — full pipeline cells will work")
else:
    print("⚠  GROQ_API_KEY not set — vector build and analysis will work; pipeline cells require it")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✓ Hugging Face login successful
✓ GROQ_API_KEY found — full pipeline cells will work


In [5]:
# ─── Cell 3 — Verify GPU ─────────────────────────────────────────────────────

import torch, transformers
print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU          : {gpu_name}")
    print(f"VRAM         : {vram_gb:.1f} GB")
    if vram_gb >= 40:
        print("✓ A100-class GPU — all model variants supported")
    elif vram_gb >= 24:
        print("✓ L4/24GB-class — Llama-3.1-8B, 3.2-3B, 7b variants supported")
    elif vram_gb >= 14:
        print("✓ T4/16GB — Llama-3.2-3B and Llama-2-7b supported")
    else:
        print(f"⚠  Only {vram_gb:.1f} GB — use Llama-3.2-1B or CPU")
else:
    print("⚠  No GPU — use Llama-3.2-1B for CPU-only inference")

torch        : 2.10.0+cu128
transformers : 5.9.0
CUDA         : True
GPU          : NVIDIA RTX 6000 Ada Generation
VRAM         : 50.9 GB
✓ A100-class GPU — all model variants supported


In [6]:
# ─── Cell 4 — SELECT MODEL ───────────────────────────────────────────────────
# Uncomment exactly ONE model.

import os, torch

# ── Model configurations ──────────────────────────────────────────────────────
MODEL_CONFIGS = {
    "meta-llama/Llama-2-7b-hf":           {"layer": 20, "alpha": 15.0, "hidden": 4096, "total_layers": 32, "vram_gb": 14.0, "format": "llama2"},
    "meta-llama/Llama-2-7b-chat-hf":      {"layer": 20, "alpha": 15.0, "hidden": 4096, "total_layers": 32, "vram_gb": 14.0, "format": "llama2chat"},
    "meta-llama/Llama-2-13b-chat-hf":     {"layer": 28, "alpha": 12.0, "hidden": 5120, "total_layers": 40, "vram_gb": 26.0, "format": "llama2chat"},
    "meta-llama/Llama-3.1-8B-Instruct":   {"layer": 20, "alpha": 13.0, "hidden": 4096, "total_layers": 32, "vram_gb": 16.0, "format": "llama3"},
    "meta-llama/Llama-3.2-3B-Instruct":   {"layer": 16, "alpha": 12.0, "hidden": 3072, "total_layers": 28, "vram_gb": 6.0,  "format": "llama3"},
    "meta-llama/Llama-3.2-1B-Instruct":   {"layer": 10, "alpha": 10.0, "hidden": 2048, "total_layers": 16, "vram_gb": 2.0,  "format": "llama3"},
}

# ── Pick ONE model ────────────────────────────────────────────────────────────
# SELECTED_MODEL = "meta-llama/Llama-2-7b-hf"            # ~14 GB
# SELECTED_MODEL = "meta-llama/Llama-2-7b-chat-hf"       # ~14 GB  (chat fine-tune)
SELECTED_MODEL = "meta-llama/Llama-2-13b-chat-hf"      # ~26 GB  (A100 required)
# SELECTED_MODEL = "meta-llama/Llama-3.1-8B-Instruct"    # ~16 GB
# SELECTED_MODEL = "meta-llama/Llama-3.2-3B-Instruct"      # ~6 GB   (recommended default)
# SELECTED_MODEL = "meta-llama/Llama-3.2-1B-Instruct"    # ~2 GB   (free T4 / CPU)

cfg = MODEL_CONFIGS[SELECTED_MODEL]

os.environ["LOCAL_MODEL_NAME"] = SELECTED_MODEL
os.environ["STEER_LAYER"]      = str(cfg["layer"])
os.environ["STEER_ALPHA"]      = str(cfg["alpha"])
os.environ["STYLE_CACHE_DIR"]  = ".style_cache"
os.environ["HF_TOKEN"]         = HF_TOKEN

depth_pct = cfg["layer"] / cfg["total_layers"] * 100

print(f"Selected model  : {SELECTED_MODEL}")
print(f"STEER_LAYER     : {cfg['layer']} / {cfg['total_layers']} ({depth_pct:.1f}% depth)")
print(f"STEER_ALPHA     : {cfg['alpha']}")
print(f"Hidden dim      : {cfg['hidden']}")
print(f"VRAM needed     : ~{cfg['vram_gb']} GB")
print(f"Prompt format   : {cfg['format']}")

Selected model  : meta-llama/Llama-2-13b-chat-hf
STEER_LAYER     : 28 / 40 (70.0% depth)
STEER_ALPHA     : 12.0
Hidden dim      : 5120
VRAM needed     : ~26.0 GB
Prompt format   : llama2chat


In [7]:
# ─── Cell 5 — Import main.py ─────────────────────────────────────────────────

import os, importlib.util
from pathlib import Path

Path('.style_cache').mkdir(exist_ok=True)

main_file = Path('main.py')
if not main_file.exists():
    raise FileNotFoundError(
        f'main.py not found in {Path.cwd()}.\n'
        'Upload main.py to the same directory as this notebook.'
    )

spec        = importlib.util.spec_from_file_location('main_module', 'main.py')
main_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(main_module)

print('✓ main.py imported successfully')
print('=' * 60)
print(f'  Model       : {os.environ["LOCAL_MODEL_NAME"]}')
print(f'  Steer layer : {os.environ["STEER_LAYER"]}')
print(f'  Steer alpha : {os.environ["STEER_ALPHA"]}')
print(f'  Cache dir   : {os.environ["STYLE_CACHE_DIR"]}')
print(f'  Contrast pairs: empathetic={len(main_module.CONTRAST_PAIRS["empathetic"])},'
      f' formal={len(main_module.CONTRAST_PAIRS["formal"])}')
print('=' * 60)

nlp_check = main_module._get_nlp()
if nlp_check:
    print('✓ spaCy en_core_web_sm loaded for KG-NER')
else:
    print('⚠  spaCy unavailable — regex-only NER fallback active')

✓ main.py imported successfully
  Model       : meta-llama/Llama-2-13b-chat-hf
  Steer layer : 28
  Steer alpha : 12.0
  Cache dir   : .style_cache
  Contrast pairs: empathetic=16, formal=16
✓ spaCy en_core_web_sm loaded for KG-NER


In [8]:
# ─── Cell 6 — Load model (GPU-first, no CPU offload) ─────────────────────────

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME     = os.environ['LOCAL_MODEL_NAME']
HF_TOKEN_      = os.environ['HF_TOKEN']
MODEL_NEEDS_GB = MODEL_CONFIGS[MODEL_NAME]['vram_gb']

if torch.cuda.is_available():
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    free_vram_gb  = (torch.cuda.get_device_properties(0).total_memory
                     - torch.cuda.memory_reserved()) / 1e9
    print(f'  GPU total VRAM  : {total_vram_gb:.1f} GB')
    print(f'  VRAM free now   : {free_vram_gb:.1f} GB')
else:
    total_vram_gb = free_vram_gb = 0

if free_vram_gb >= MODEL_NEEDS_GB:
    device_map_arg = {'': 0}
    max_memory_arg = None
    print('  Strategy : GPU-only — no CPU offload')
elif '1B' in MODEL_NAME and total_vram_gb < MODEL_NEEDS_GB:
    device_map_arg = 'cpu'
    max_memory_arg = None
    print('  Strategy : CPU fallback for 1B model')
else:
    gpu_limit      = f'{int(total_vram_gb - 1)}GiB'
    device_map_arg = 'auto'
    max_memory_arg = {0: gpu_limit, 'cpu': '4GiB'}
    print(f'  Strategy : auto split — GPU {gpu_limit} + cpu 4GiB')

print(f'\nLoading {MODEL_NAME} ...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=HF_TOKEN_)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

_load_kwargs = dict(
    trust_remote_code=True, torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True, device_map=device_map_arg, token=HF_TOKEN_,
)
if max_memory_arg is not None:
    _load_kwargs['max_memory'] = max_memory_arg

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **_load_kwargs)
model.eval()

main_module._model_cache['model']     = model
main_module._model_cache['tokenizer'] = tokenizer

if torch.cuda.is_available() and device_map_arg != 'cpu':
    vram_used  = torch.cuda.memory_allocated() / 1e9
    cpu_params = sum(1 for p in model.parameters() if p.device.type in ('cpu', 'meta'))
    gpu_params = sum(1 for p in model.parameters() if p.device.type == 'cuda')
    print(f'\n✓ {MODEL_NAME} loaded')
    print(f'  VRAM allocated : {vram_used:.2f} GB')
    print(f'  GPU params     : {gpu_params}')
    print(f'  CPU/meta params: {cpu_params}  {"⚠" if cpu_params else "✓"}')
else:
    print(f'\n✓ {MODEL_NAME} loaded on CPU')

n_layers = (len(model.model.layers)
            if hasattr(model, 'model') and hasattr(model.model, 'layers') else '?')
hidden   = model.config.hidden_size if hasattr(model, 'config') else '?'
print(f'  Layers     : {n_layers}  (STEER_LAYER={os.environ["STEER_LAYER"]})')
print(f'  Hidden dim : {hidden}')

  GPU total VRAM  : 50.9 GB
  VRAM free now   : 50.9 GB
  Strategy : GPU-only — no CPU offload

Loading meta-llama/Llama-2-13b-chat-hf ...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]


✓ meta-llama/Llama-2-13b-chat-hf loaded
  VRAM allocated : 26.03 GB
  GPU params     : 363
  CPU/meta params: 0  ✓
  Layers     : 40  (STEER_LAYER=28)
  Hidden dim : 5120


---
## 🔍 Steering Layer Analysis

Why does layer choice matter? Transformer hidden states evolve across layers:
- **Early layers (0–30%)**: Token-level patterns, positional encoding, syntax.
- **Middle layers (30–60%)**: Semantic composition, entity resolution.
- **Middle-late layers (60–70%)**: **Style and pragmatics are encoded here** — this is the sweet spot for steering.
- **Late layers (70–100%)**: Output vocabulary projection, already committed to token predictions.

The cells below probe activation geometry across layers to confirm the chosen `STEER_LAYER` captures the most separable style signal.

In [9]:
# ─── Cell 7 — Patch activation capture for device_map='auto' safety ──────────

import torch

def _patched_get_activation(model, tokenizer, text: str, layer_idx: int = None):
    """
    Captures last-token hidden state at a given layer index (defaults to STEER_LAYER).
    Device-safe for device_map='auto': resolves first real (non-meta) parameter device.
    """
    if layer_idx is None:
        layer_idx = int(os.environ.get('STEER_LAYER', cfg['layer']))

    captured = {}

    # Get the specific layer
    if hasattr(model, 'model') and hasattr(model.model, 'layers'):
        target_layer = model.model.layers[layer_idx]
    else:
        raise ValueError("Model architecture not supported — cannot find model.model.layers")

    def _hook(module, inp, output):
        hidden = output[0] if isinstance(output, tuple) else output
        if hidden.device.type != 'meta':
            captured["act"] = hidden.detach().cpu().float()

    handle = target_layer.register_forward_hook(_hook)
    try:
        real_device = next(
            (p.device for p in model.parameters() if p.device.type != 'meta'),
            torch.device('cpu')
        )
        fmt = MODEL_CONFIGS[MODEL_NAME]['format']
        add_sp = fmt != 'llama3'
        inputs = tokenizer(text, return_tensors='pt', padding=False,
                           return_attention_mask=True,
                           add_special_tokens=add_sp).to(real_device)
        with torch.no_grad():
            model(**inputs)
    finally:
        handle.remove()

    act = captured.get('act')
    if act is None:
        raise RuntimeError('Hook did not capture activations — all params on meta device?')
    return act[0, -1, :]


def _patched_run_kg_activation_steering(defactualized_prompt: str, style: str, kg) -> str:
    """
    Device-safe steering hook. vec.to(hidden.device) inside hook = never a device mismatch.
    Supports both Llama-2 (chat + base) and Llama-3.x prompt formats.
    """
    steer_vec = main_module.get_style_vector(style)
    if steer_vec is None:
        raise RuntimeError(f"No style vector for '{style}'. Build it first.")

    model_, tokenizer_ = main_module._get_model_and_tokenizer()
    real_device = next(
        (p.device for p in model_.parameters() if p.device.type != 'meta'),
        torch.device('cuda:0')
    )

    style_prefix = {
        "empathetic": "I'm truly sorry to hear about this and I completely understand how frustrated you must be feeling right now.",
        "formal":     "We acknowledge receipt of your complaint and wish to advise you that your case has been logged and assigned for review.",
    }.get(style, "Thank you for contacting us regarding this matter.")

    system_instruction = (
        f"You are a professional customer support agent. "
        f"Write a {style.upper()} support reply. "
        "Do NOT use headers or bullet points. "
        "Write ONLY the reply — 3 to 4 sentences maximum."
    )

    fmt = MODEL_CONFIGS[MODEL_NAME]['format']
    if fmt == 'llama3':
        full_prompt = (
            '<|begin_of_text|>'
            '<|start_header_id|>system<|end_header_id|>\n\n'
            f'{system_instruction}<|eot_id|>'
            '<|start_header_id|>user<|end_header_id|>\n\n'
            f'Customer message: {defactualized_prompt}<|eot_id|>'
            '<|start_header_id|>assistant<|end_header_id|>\n\n'
            f'{style_prefix}'
        )
        add_sp = False
    else:  # llama2 / llama2chat
        full_prompt = (
            f'<s>[INST] <<SYS>>\n{system_instruction}\n<</SYS>>\n\n'
            f'Customer message: {defactualized_prompt} [/INST] {style_prefix}'
        )
        add_sp = True

    inputs    = tokenizer_(full_prompt, return_tensors='pt', padding=False,
                            return_attention_mask=True,
                            add_special_tokens=add_sp).to(real_device)
    input_ids = inputs.input_ids
    STEER_ALPHA  = main_module.STEER_ALPHA
    target_layer = main_module._get_layer(model_)

    def _steer_hook(module, inp, output):
        hidden = output[0] if isinstance(output, tuple) else output
        if hidden.device.type == 'meta':
            return output
        h_f32  = hidden.float()
        v      = steer_vec.to(device=h_f32.device, dtype=torch.float32)
        h_f32  = h_f32 + STEER_ALPHA * v
        steered = h_f32.to(hidden.dtype)
        return (steered,) + output[1:] if isinstance(output, tuple) else steered

    # Build EOS list (Llama-3.x uses dual EOS)
    _eos_ids = [tokenizer_.eos_token_id]
    if fmt == 'llama3':
        _eot_id = tokenizer_.convert_tokens_to_ids('<|eot_id|>')
        if _eot_id and _eot_id not in _eos_ids:
            _eos_ids.append(_eot_id)

    handle = target_layer.register_forward_hook(_steer_hook)
    try:
        with torch.no_grad():
            output_ids = model_.generate(
                input_ids, attention_mask=inputs.attention_mask,
                max_new_tokens=120, do_sample=False,
                pad_token_id=tokenizer_.eos_token_id,
                repetition_penalty=1.3, eos_token_id=_eos_ids,
            )
    finally:
        handle.remove()

    new_ids   = output_ids[0][input_ids.shape[1]:]
    result    = tokenizer_.decode(new_ids, skip_special_tokens=True).strip()
    sentences = [s.strip() for s in result.split('.') if s.strip()]
    if len(sentences) > 4:
        result = '. '.join(sentences[:4]) + '.'
    return result


main_module._get_activation             = _patched_get_activation
main_module._run_kg_activation_steering = _patched_run_kg_activation_steering
print("✓ Patches applied: _get_activation + _run_kg_activation_steering")
print("  Both are device_map='auto' safe and support all prompt formats.")

✓ Patches applied: _get_activation + _run_kg_activation_steering
  Both are device_map='auto' safe and support all prompt formats.


In [10]:
# ─── Cell 8 — Layer-wise Style Separability Probe ────────────────────────────
#
# For each candidate steering layer, compute the cosine similarity between
# mean empathetic activations and mean formal activations.
# Lower cosine sim (more orthogonal / negative) = better separability.
# The chosen STEER_LAYER should have the most negative / lowest similarity.

import torch
import numpy as np
import matplotlib
matplotlib.use('Agg')  # for Colab/headless
import matplotlib.pyplot as plt

PROBE_TEXTS = {
    "empathetic": [
        "I completely understand how frustrating this must be for you.",
        "I'm so sorry to hear about your experience — we want to make this right.",
        "Your feelings are completely valid, and I genuinely want to help resolve this.",
        "I hear you, and I deeply apologize for the inconvenience caused.",
    ],
    "formal": [
        "We acknowledge your complaint and have initiated a review of your case.",
        "Your request has been logged and assigned a reference number.",
        "We regret any inconvenience. The matter is under investigation per protocol.",
        "This correspondence confirms receipt of your escalation request.",
    ],
}

n_layers  = len(model.model.layers)
# Probe every 2nd layer to save time; adjust step for thoroughness
STEP      = max(1, n_layers // 14)
probe_ids = list(range(0, n_layers, STEP))

print(f"Probing {len(probe_ids)} layers (step={STEP}) out of {n_layers} total ...")
print("(Each layer: 8 forward passes — ~30 sec on T4)")
print()

layer_sims   = []
layer_norms  = []

for i, layer_idx in enumerate(probe_ids):
    print(f"  Layer {layer_idx:2d} ({i+1}/{len(probe_ids)}) ...", end="\r")
    emp_acts  = [_patched_get_activation(model, tokenizer, t, layer_idx=layer_idx)
                 for t in PROBE_TEXTS["empathetic"]]
    form_acts = [_patched_get_activation(model, tokenizer, t, layer_idx=layer_idx)
                 for t in PROBE_TEXTS["formal"]]

    mean_emp  = torch.stack(emp_acts).mean(0)
    mean_form = torch.stack(form_acts).mean(0)
    mean_emp  = mean_emp  / mean_emp.norm()
    mean_form = mean_form / mean_form.norm()

    sim  = torch.nn.functional.cosine_similarity(mean_emp.unsqueeze(0), mean_form.unsqueeze(0)).item()
    norm = (mean_emp - mean_form).norm().item()
    layer_sims.append(sim)
    layer_norms.append(norm)

print(f"  Done — probed {len(probe_ids)} layers.          ")

# ── Plot ────────────────────────────────────────────────────────────────────
steer_layer = cfg['layer']
fig, axes   = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(probe_ids, layer_sims, 'b-o', markersize=4)
axes[0].axvline(steer_layer, color='red', linestyle='--', linewidth=2, label=f'STEER_LAYER={steer_layer}')
axes[0].set_xlabel('Layer index')
axes[0].set_ylabel('Cosine similarity (emp vs formal)')
axes[0].set_title('Style Separability by Layer\n(lower = more separable)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(probe_ids, layer_norms, 'g-s', markersize=4)
axes[1].axvline(steer_layer, color='red', linestyle='--', linewidth=2, label=f'STEER_LAYER={steer_layer}')
axes[1].set_xlabel('Layer index')
axes[1].set_ylabel('L2 distance (emp vs formal)')
axes[1].set_title('Style Direction Gap by Layer\n(higher = more separable)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'{MODEL_NAME} — Layer Steering Analysis', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('steering_layer_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print('\n✓ Plot saved: steering_layer_analysis.png')

# ── Report ──────────────────────────────────────────────────────────────────
best_idx  = int(np.argmin(layer_sims))
best_layer = probe_ids[best_idx]
chosen_sim = layer_sims[probe_ids.index(steer_layer)] if steer_layer in probe_ids else None

print(f"\n{'='*55}")
print(f"  Most separable layer (min cosine sim): {best_layer}  (sim={layer_sims[best_idx]:.4f})")
print(f"  Configured STEER_LAYER               : {steer_layer}")
if chosen_sim is not None:
    print(f"  Cosine sim @ STEER_LAYER             : {chosen_sim:.4f}")
print(f"{'='*55}")

Probing 20 layers (step=2) out of 40 total ...
(Each layer: 8 forward passes — ~30 sec on T4)

  Done — probed 20 layers.          

✓ Plot saved: steering_layer_analysis.png

  Most separable layer (min cosine sim): 14  (sim=0.5646)
  Configured STEER_LAYER               : 28
  Cosine sim @ STEER_LAYER             : 0.6793


In [11]:
# ─── Cell 9 — Build EMPATHETIC style vector ──────────────────────────────────

print('Building EMPATHETIC vector (16 contrast pairs × 2 activations)...')
vec_emp = main_module.build_style_vector(style='empathetic', method='pca')
print(f'\n✓ EMPATHETIC vector built')
print(f'  shape : {list(vec_emp.shape)}')
print(f'  norm  : {vec_emp.norm().item():.6f}  (should be ~1.0)')
print(f'  dtype : {vec_emp.dtype}')

Building EMPATHETIC vector (16 contrast pairs × 2 activations)...

✓ EMPATHETIC vector built
  shape : [5120]
  norm  : 0.999999  (should be ~1.0)
  dtype : torch.float32


In [12]:
# ─── Cell 10 — Build FORMAL style vector ────────────────────────────────────

print('Building FORMAL vector (model already cached)...')
vec_form = main_module.build_style_vector(style='formal', method='pca')
print(f'\n✓ FORMAL vector built')
print(f'  shape : {list(vec_form.shape)}')
print(f'  norm  : {vec_form.norm().item():.6f}  (should be ~1.0)')
print(f'  dtype : {vec_form.dtype}')

Building FORMAL vector (model already cached)...



✓ FORMAL vector built
  shape : [5120]
  norm  : 0.999999  (should be ~1.0)
  dtype : torch.float32


In [13]:
# ─── Cell 11 — Verify both vectors ───────────────────────────────────────────

import torch
HIDDEN_DIM = cfg['hidden']

sim = torch.nn.functional.cosine_similarity(
    vec_emp.unsqueeze(0), vec_form.unsqueeze(0)
).item()

print('=' * 55)
print(f'  empathetic : norm={vec_emp.norm():.6f}  shape={list(vec_emp.shape)}')
print(f'  formal     : norm={vec_form.norm():.6f}  shape={list(vec_form.shape)}')
print(f'  cosine sim : {sim:.4f}')
print('=' * 55)

assert abs(vec_emp.norm().item()  - 1.0) < 0.01, f'empathetic not unit-norm! {vec_emp.norm():.4f}'
assert abs(vec_form.norm().item() - 1.0) < 0.01, f'formal not unit-norm! {vec_form.norm():.4f}'
assert vec_emp.shape[0]  == HIDDEN_DIM, f'Expected {HIDDEN_DIM}, got {vec_emp.shape[0]}'
assert vec_form.shape[0] == HIDDEN_DIM, f'Expected {HIDDEN_DIM}, got {vec_form.shape[0]}'
assert sim < 0.5, f'Cosine sim {sim:.3f} too high — vectors may not be distinct!'

print()
if sim < 0:
    print('✓ Cosine similarity NEGATIVE — vectors point in opposite directions (ideal)')
else:
    print(f'  sim={sim:.3f} — vectors diverge but are not anti-parallel (acceptable if <0.5)')

import os
cache_files = sorted(os.listdir('.style_cache'))
print('\nCached files in .style_cache/:')
for f in cache_files:
    size = os.path.getsize(f'.style_cache/{f}') / 1024
    print(f'  {f}  ({size:.1f} KB)')
print('\n✓ All assertions passed!')

  empathetic : norm=0.999999  shape=[5120]
  formal     : norm=0.999999  shape=[5120]
  cosine sim : -0.9781

✓ Cosine similarity NEGATIVE — vectors point in opposite directions (ideal)

Cached files in .style_cache/:
  style_vec_empathetic_pca.pkl  (20.4 KB)
  style_vec_formal_pca.pkl  (20.4 KB)

✓ All assertions passed!


---
## 📊 Key Metrics

### Steering Activation Fidelity (SAF)
Measures how much of the intended style direction survives in the output hidden state after steering.
- Formula: `SAF = cosine_sim(output_hidden, style_vec)` — higher is better (range: −1 to 1).

### Hallucination Severity Index (HSI)
Measures entity leakage or fabrication: how many entity tokens in the model's response were NOT present in the masked (defactualized) input.
- Formula: `HSI = |spurious_entities| / |response_entities|` — lower is better (range: 0 to 1).

### Tone Consistency Index (TCI)
Measures directional alignment of the response's hidden state with the target style vector.
- Formula: `TCI = (SAF_target - SAF_opposite) / 2` — higher is better (range: −1 to 1).

In [21]:
# ─── Cell 12 — Compute SAF, HSI, TCI ────────────────────────────────────────

import torch
import re
import spacy
from typing import Dict, List

# Load spaCy for entity extraction
try:
    nlp = spacy.load('en_core_web_sm')
    _SPACY_AVAILABLE = True
except Exception:
    _SPACY_AVAILABLE = False
    print('⚠  spaCy not available — HSI will use whitelist-based entity detection')


def compute_saf(text: str, style_vec: torch.Tensor, model, tokenizer,
                layer_idx: int = None) -> float:
    """
    Steering Activation Fidelity:
    Cosine similarity between the output hidden state at STEER_LAYER and the style vector.
    Higher = steer vector direction is better represented in the output.
    """
    act  = _patched_get_activation(model, tokenizer, text, layer_idx=layer_idx)
    act_n = act / act.norm()
    saf   = torch.nn.functional.cosine_similarity(
        act_n.unsqueeze(0), style_vec.unsqueeze(0)
    ).item()
    return saf


def compute_hsi(input_text: str, output_text: str) -> Dict:
    """
    Hallucination Severity Index:
    Ratio of output entities not grounded in the input.
    Lower = fewer hallucinated / leaked entities.
    """
    if _SPACY_AVAILABLE:
        in_ents  = {e.text.lower() for e in nlp(input_text).ents}
        out_ents = {e.text.lower() for e in nlp(output_text).ents}
    else:
        # Fallback: capitalised words as proxy entities
        in_ents  = set(re.findall(r'\b[A-Z][a-z]+\b', input_text))
        out_ents = set(re.findall(r'\b[A-Z][a-z]+\b', output_text))

    spurious = out_ents - in_ents
    hsi      = len(spurious) / max(len(out_ents), 1)
    return {
        'hsi':             round(hsi, 4),
        'input_entities':  sorted(in_ents),
        'output_entities': sorted(out_ents),
        'spurious':        sorted(spurious),
    }


def compute_tci(output_text: str, target_vec: torch.Tensor,
                opposite_vec: torch.Tensor, model, tokenizer) -> float:
    """
    Tone Consistency Index:
    How much the output aligns with the target style relative to the opposite.
    TCI = (SAF_target - SAF_opposite) / 2   →   range [-1, 1], higher is better.
    """
    saf_target   = compute_saf(output_text, target_vec,   model, tokenizer)
    saf_opposite = compute_saf(output_text, opposite_vec, model, tokenizer)
    tci = (saf_target - saf_opposite) / 2.0
    return round(tci, 4)


print('✓ SAF, HSI, TCI functions defined')
print()
print('  compute_saf(text, style_vec, model, tokenizer)  → float  [-1, 1]  (higher=better)')
print('  compute_hsi(input_text, output_text)            → dict   {hsi, spurious, ...}')
print('  compute_tci(output_text, target_vec, opp_vec, model, tokenizer)  → float  [-1, 1]')

✓ SAF, HSI, TCI functions defined

  compute_saf(text, style_vec, model, tokenizer)  → float  [-1, 1]  (higher=better)
  compute_hsi(input_text, output_text)            → dict   {hsi, spurious, ...}
  compute_tci(output_text, target_vec, opp_vec, model, tokenizer)  → float  [-1, 1]


---
## 🧪 Tests

In [23]:
# ─── Cell 13 — Smoke Test: Two steering vectors, different tones ─────────

import torch

# ─── Safe steering config resolution ────────────────────────────────────────
try:
    _, ALPHA_TEST = main_module._resolve_steer_config()
    print(f"✓ Steering config loaded: ALPHA_TEST = {ALPHA_TEST}")
except AttributeError:
    ALPHA_TEST = 0.5  # Default
    print(f"⚠ _resolve_steer_config not found, using default: {ALPHA_TEST}")

# Get model and tokenizer
try:
    model_t, tok_t = main_module._get_model_and_tokenizer()
    print("✓ Model and tokenizer loaded")
except AttributeError as e:
    print(f"✗ Error loading model/tokenizer: {e}")
    raise

# ─── Setup EOS token IDs ───────────────────────────────────────────────────
_eos_ids = [tok_t.eos_token_id]

# Get model format with fallback
try:
    fmt = MODEL_CONFIGS[MODEL_NAME]['format']
except (NameError, KeyError):
    fmt = 'llama2'  # Default
    print(f"⚠ MODEL_CONFIGS not found, using default: {fmt}")

if fmt == 'llama3':
    try:
        _eot_id = tok_t.convert_tokens_to_ids('<|eot_id|>')
        if _eot_id and _eot_id not in _eos_ids:
            _eos_ids.append(_eot_id)
        print(f"✓ Llama3 format: added eot_id")
    except Exception as e:
        print(f"⚠ Could not set up Llama3 eot_id: {e}")

# System message
system_msg = (
    'You are a professional customer support agent. '
    'Write a support reply. Do NOT use headers or bullet points. '
    'Write ONLY the reply — 3 to 4 sentences maximum.'
)

# Build prompt based on format
if fmt == 'llama3':
    test_prompt = (
        '<|begin_of_text|>'
        '<|start_header_id|>system<|end_header_id|>\n\n'
        f'{system_msg}<|eot_id|>'
        '<|start_header_id|>user<|end_header_id|>\n\n'
        'Customer message: My <PRODUCT> order <ORDER_ID> has an <ISSUE>.<|eot_id|>'
        '<|start_header_id|>assistant<|end_header_id|>\n\n'
    )
    add_sp = False
else:
    test_prompt = (
        f'<s>[INST] <<SYS>>\n{system_msg}\n<</SYS>>\n\n'
        'Customer message: My <PRODUCT> order <ORDER_ID> has an <ISSUE>. [/INST] '
    )
    add_sp = True

# Generation function with full error handling
def generate_steered(prompt, style_vec, alpha, label, add_special_tokens=False):
    """Generate text with steering vector applied."""
    try:
        real_device = next(
            (p.device for p in model_t.parameters() if p.device.type != 'meta'),
            torch.device('cpu')
        )
        
        vec = style_vec.to(device=real_device, dtype=torch.float32)
        
        try:
            layer = main_module._get_layer(model_t)
        except AttributeError:
            print(f"⚠ {label}: _get_layer not found, skipping steering")
            return "Error: steering layer not available"
        
        def _hook(module, inp, output):
            h = output[0] if isinstance(output, tuple) else output
            if h.device.type == 'meta': return output
            h_f32 = h.float() + alpha * vec.to(h.device)
            steered = h_f32.to(h.dtype)
            return (steered,) + output[1:] if isinstance(output, tuple) else steered
        
        handle = layer.register_forward_hook(_hook)
        
        try:
            inputs = tok_t(prompt, return_tensors='pt',
                          add_special_tokens=add_special_tokens).to(real_device)
            with torch.no_grad():
                out = model_t.generate(
                    inputs.input_ids,
                    max_new_tokens=80,
                    do_sample=False,
                    pad_token_id=tok_t.eos_token_id,
                    repetition_penalty=1.3,
                    eos_token_id=_eos_ids,
                )
            
            new_ids = out[0][inputs.input_ids.shape[1]:]
            text = tok_t.decode(new_ids, skip_special_tokens=True).strip()
            
            print(f'\n[{label}]\n  {text[:300]}')
            return text
            
        finally:
            handle.remove()
    
    except Exception as e:
        print(f'\n[{label}]\n  ✗ Error: {str(e)[:100]}')
        return ""

# ─── Run smoke test ───────────────────────────────────────────────────────
print('=' * 70)
print('SMOKE TEST: Same prompt — Two steering vectors — Different tones')
print('=' * 70)

# Validate steering vectors exist
try:
    assert 'vec_emp' in dir(), "vec_emp not defined. Run Cell 12 first."
    assert 'vec_form' in dir(), "vec_form not defined. Run Cell 12 first."
    print("✓ Steering vectors available")
except AssertionError as e:
    print(f"✗ {e}")
    raise

print(f"\nAlpha: {ALPHA_TEST}")
r_emp = generate_steered(test_prompt, vec_emp, ALPHA_TEST, 'EMPATHETIC', add_sp)
r_form = generate_steered(test_prompt, vec_form, ALPHA_TEST, 'FORMAL', add_sp)

print('\n' + '=' * 70)
print('✓ Smoke test complete')
print("  EMPATHETIC → warm, personal ('I'm sorry', 'I understand')")
print("  FORMAL     → institutional ('We acknowledge', 'has been logged')")
print('=' * 70)

[transformers] Both `max_new_tokens` (=80) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


⚠ _resolve_steer_config not found, using default: 0.5
✓ Model and tokenizer loaded
SMOKE TEST: Same prompt — Two steering vectors — Different tones
✓ Steering vectors available

Alpha: 0.5


[transformers] Both `max_new_tokens` (=80) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[EMPATHETIC]
  Sure thing! Here's my response as a professional customer support agent:

Thank you for reaching out about your recent purchase of our product with order ID #<ORDER_ID>. Sorry to hear that there was an issue with your order - can you please provide more details regarding what specifically went wrong

[FORMAL]
  Sure thing! Here's my response as a professional customer support agent:

Thank you for reaching out about your recent purchase of our product with order ID #<ORDER_ID>. Sorry to hear that there was an issue - could you please provide more details regarding what happened so we can assist you better?

✓ Smoke test complete
  EMPATHETIC → warm, personal ('I'm sorry', 'I understand')
  FORMAL     → institutional ('We acknowledge', 'has been logged')


In [24]:
# ─── Cell 14 — Metric Evaluation: SAF + TCI on smoke-test outputs ────────────

import torch

print('Computing SAF, HSI, TCI on smoke-test outputs...')
print('=' * 60)

# ── Steering Activation Fidelity ─────────────────────────────────────────────
saf_emp_target    = compute_saf(r_emp,  vec_emp,  model_t, tok_t)
saf_emp_opposite  = compute_saf(r_emp,  vec_form, model_t, tok_t)
saf_form_target   = compute_saf(r_form, vec_form, model_t, tok_t)
saf_form_opposite = compute_saf(r_form, vec_emp,  model_t, tok_t)

print(f'\n  Steering Activation Fidelity (SAF)  [higher = steer direction more present in output]')
print(f'  EMPATHETIC output vs emp vec : {saf_emp_target:+.4f}  ← target')
print(f'  EMPATHETIC output vs form vec: {saf_emp_opposite:+.4f}  ← opposite')
print(f'  FORMAL output vs form vec    : {saf_form_target:+.4f}  ← target')
print(f'  FORMAL output vs emp vec     : {saf_form_opposite:+.4f}  ← opposite')

# ── Tone Consistency Index ────────────────────────────────────────────────────
tci_emp  = (saf_emp_target  - saf_emp_opposite)  / 2.0
tci_form = (saf_form_target - saf_form_opposite) / 2.0

print(f'\n  Tone Consistency Index (TCI)  [higher = output better aligned with target style]')
print(f'  EMPATHETIC TCI : {tci_emp:+.4f}')
print(f'  FORMAL TCI     : {tci_form:+.4f}')

# ── Hallucination Severity Index ─────────────────────────────────────────────
INPUT_TEXT = 'My <PRODUCT> order <ORDER_ID> has an <ISSUE>.'
hsi_emp  = compute_hsi(INPUT_TEXT, r_emp)
hsi_form = compute_hsi(INPUT_TEXT, r_form)

print(f'\n  Hallucination Severity Index (HSI)  [lower = fewer spurious entities]')
print(f'  EMPATHETIC HSI : {hsi_emp["hsi"]:.4f}')
print(f'    Spurious ents: {hsi_emp["spurious"]}')
print(f'  FORMAL HSI     : {hsi_form["hsi"]:.4f}')
print(f'    Spurious ents: {hsi_form["spurious"]}')

print('=' * 60)
print('✓ Metrics computed')

Computing SAF, HSI, TCI on smoke-test outputs...

  Steering Activation Fidelity (SAF)  [higher = steer direction more present in output]
  EMPATHETIC output vs emp vec : +0.0181  ← target
  EMPATHETIC output vs form vec: -0.0193  ← opposite
  FORMAL output vs form vec    : +0.0400  ← target
  FORMAL output vs emp vec     : -0.0264  ← opposite

  Tone Consistency Index (TCI)  [higher = output better aligned with target style]
  EMPATHETIC TCI : +0.0187
  FORMAL TCI     : +0.0332

  Hallucination Severity Index (HSI)  [lower = fewer spurious entities]
  EMPATHETIC HSI : 0.0000
    Spurious ents: []
  FORMAL HSI     : 0.0000
    Spurious ents: []
✓ Metrics computed


In [25]:
# ─── Cell 15 — Alpha Sweep: find optimal STEER_ALPHA ────────────────────────
#
# TCI peaks at a sweet spot: too low and the style doesn't take hold;
# too high and the model becomes incoherent (HSI spikes).
# Run this to validate the configured alpha and see where the optimum lies.

import torch
import numpy as np

TEST_ALPHAS = [4.0, 6.0, 8.0, 10.0, 12.0, 14.0, 16.0, 18.0]
SWEEP_INPUT = 'My <PRODUCT> order <ORDER_ID> has an <ISSUE>.'

results = {'alpha': [], 'tci_emp': [], 'tci_form': [], 'hsi_emp': [], 'hsi_form': []}

real_device = next(
    (p.device for p in model_t.parameters() if p.device.type != 'meta'),
    torch.device('cpu')
)

print(f'Alpha sweep over {TEST_ALPHAS} ...')
for alpha in TEST_ALPHAS:
    print(f'  alpha={alpha} ...', end='\r')

    def _gen(style_vec):
        vec   = style_vec.to(device=real_device, dtype=torch.float32)
        layer = main_module._get_layer(model_t)
        def _hook(m, inp, out):
            h = out[0] if isinstance(out, tuple) else out
            if h.device.type == 'meta': return out
            h_f32 = h.float() + alpha * vec.to(h.device)
            s = h_f32.to(h.dtype)
            return (s,) + out[1:] if isinstance(out, tuple) else s
        handle = layer.register_forward_hook(_hook)
        inputs = tok_t(test_prompt, return_tensors='pt',
                       add_special_tokens=add_sp).to(real_device)
        try:
            with torch.no_grad():
                out = model_t.generate(inputs.input_ids, max_new_tokens=60,
                                       do_sample=False, pad_token_id=tok_t.eos_token_id,
                                       repetition_penalty=1.3, eos_token_id=_eos_ids)
        finally:
            handle.remove()
        new_ids = out[0][inputs.input_ids.shape[1]:]
        return tok_t.decode(new_ids, skip_special_tokens=True).strip()

    out_e = _gen(vec_emp)
    out_f = _gen(vec_form)

    saf_et = compute_saf(out_e, vec_emp,  model_t, tok_t)
    saf_eo = compute_saf(out_e, vec_form, model_t, tok_t)
    saf_ft = compute_saf(out_f, vec_form, model_t, tok_t)
    saf_fo = compute_saf(out_f, vec_emp,  model_t, tok_t)

    results['alpha'].append(alpha)
    results['tci_emp'].append((saf_et - saf_eo) / 2)
    results['tci_form'].append((saf_ft - saf_fo) / 2)
    results['hsi_emp'].append(compute_hsi(SWEEP_INPUT, out_e)['hsi'])
    results['hsi_form'].append(compute_hsi(SWEEP_INPUT, out_f)['hsi'])

print('  Done.                          ')

# ── Plot ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(results['alpha'], results['tci_emp'],  'b-o', label='TCI — Empathetic')
axes[0].plot(results['alpha'], results['tci_form'], 'r-s', label='TCI — Formal')
axes[0].axvline(cfg['alpha'], color='green', linestyle='--', label=f"Configured α={cfg['alpha']}")
axes[0].set_xlabel('STEER_ALPHA')
axes[0].set_ylabel('Tone Consistency Index (TCI)')
axes[0].set_title('TCI vs Alpha')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(results['alpha'], results['hsi_emp'],  'b-o', label='HSI — Empathetic')
axes[1].plot(results['alpha'], results['hsi_form'], 'r-s', label='HSI — Formal')
axes[1].axvline(cfg['alpha'], color='green', linestyle='--', label=f"Configured α={cfg['alpha']}")
axes[1].set_xlabel('STEER_ALPHA')
axes[1].set_ylabel('Hallucination Severity Index (HSI)')
axes[1].set_title('HSI vs Alpha (lower is better)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle(f'{MODEL_NAME} — Alpha Sweep', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('alpha_sweep.png', dpi=120, bbox_inches='tight')
plt.show()

best_alpha_emp  = results['alpha'][int(np.argmax(results['tci_emp']))]
best_alpha_form = results['alpha'][int(np.argmax(results['tci_form']))]
print(f'\n  Best alpha for EMPATHETIC TCI : {best_alpha_emp}')
print(f'  Best alpha for FORMAL TCI     : {best_alpha_form}')
print(f'  Configured STEER_ALPHA        : {cfg["alpha"]}')

[transformers] Both `max_new_tokens` (=60) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Alpha sweep over [4.0, 6.0, 8.0, 10.0, 12.0, 14.0, 16.0, 18.0] ...


[transformers] Both `max_new_tokens` (=60) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=60) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=60) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[transformers] Both `max_new_tokens` (=60) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=60) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=60) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=60) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Done.                          

  Best alpha for EMPATHETIC TCI : 12.0
  Best alpha for FORMAL TCI     : 10.0
  Configured STEER_ALPHA        : 12.0


In [26]:
# ─── Cell 16 — Multi-scenario Metric Test ────────────────────────────────────
#
# Runs 4 distinct complaint scenarios through both style vectors.
# Reports SAF, TCI, HSI per scenario and a final aggregated summary.

import torch

SCENARIOS = [
    {"id": "battery_issue",  "input": "My <PRODUCT> battery drains in 2 hours. Order <ORDER_ID>."},
    {"id": "wrong_item",     "input": "I received the wrong item in order <ORDER_ID>. I ordered <PRODUCT>."},
    {"id": "billing_error",  "input": "I was charged twice for order <ORDER_ID>. Please refund <AMOUNT>."},
    {"id": "delivery_delay", "input": "Order <ORDER_ID> was supposed to arrive 5 days ago. Where is my <PRODUCT>?"},
]

print(f'Running {len(SCENARIOS)} scenarios × 2 styles = {len(SCENARIOS)*2} generations...')
print('=' * 70)

all_records = []

for sc in SCENARIOS:
    sc_input = sc['input']
    if fmt == 'llama3':
        sc_prompt = (
            '<|begin_of_text|>'
            '<|start_header_id|>system<|end_header_id|>\n\n'
            f'{system_msg}<|eot_id|>'
            '<|start_header_id|>user<|end_header_id|>\n\n'
            f'Customer message: {sc_input}<|eot_id|>'
            '<|start_header_id|>assistant<|end_header_id|>\n\n'
        )
    else:
        sc_prompt = (
            f'<s>[INST] <<SYS>>\n{system_msg}\n<</SYS>>\n\n'
            f'Customer message: {sc_input} [/INST] '
        )

    for style_label, style_vec in [('empathetic', vec_emp), ('formal', vec_form)]:
        opp_vec = vec_form if style_label == 'empathetic' else vec_emp
        out = generate_steered(sc_prompt, style_vec, ALPHA_TEST, f"{sc['id']}/{style_label}", add_sp)

        saf = compute_saf(out, style_vec, model_t, tok_t)
        tci = compute_tci(out, style_vec, opp_vec,  model_t, tok_t)
        hsi = compute_hsi(sc_input, out)

        all_records.append({
            'scenario': sc['id'], 'style': style_label,
            'SAF': round(saf, 4), 'TCI': round(tci, 4), 'HSI': hsi['hsi'],
            'spurious_entities': hsi['spurious'],
        })

# ── Summary Table ─────────────────────────────────────────────────────────────
print('\n' + '=' * 70)
print(f'  {"Scenario":<20} {"Style":<12} {"SAF":>7} {"TCI":>7} {"HSI":>7}  Spurious')
print('-' * 70)
for r in all_records:
    sp = ', '.join(r['spurious_entities']) if r['spurious_entities'] else '—'
    print(f'  {r["scenario"]:<20} {r["style"]:<12} '
          f'{r["SAF"]:>7.4f} {r["TCI"]:>7.4f} {r["HSI"]:>7.4f}  {sp}')

print('=' * 70)

# ── Aggregated averages ──────────────────────────────────────────────────────
import statistics
for style in ['empathetic', 'formal']:
    rows = [r for r in all_records if r['style'] == style]
    mean_saf = statistics.mean(r['SAF'] for r in rows)
    mean_tci = statistics.mean(r['TCI'] for r in rows)
    mean_hsi = statistics.mean(r['HSI'] for r in rows)
    print(f'  AVG {style:<12}: SAF={mean_saf:+.4f}  TCI={mean_tci:+.4f}  HSI={mean_hsi:.4f}')

print('\n✓ Multi-scenario test complete')

[transformers] Both `max_new_tokens` (=80) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running 4 scenarios × 2 styles = 8 generations...

[battery_issue/empathetic]
  Sure thing! I'd be happy to help you with your concern about your <PRODUCT> battery life. Based on your order ID, it looks like you received a replacement device due to previous issues with rapid battery drainage. Our team has tested this new unit and confirmed that it should provide improved perfor


[transformers] Both `max_new_tokens` (=80) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[battery_issue/formal]
  Sure thing! I'd be happy to help you with your concern about your <PRODUCT> battery life. Based on your order ID, it looks like you received a replacement device due to previous issues with rapid battery drainage. Our team has tested this new unit and confirmed that it should provide improved perfor


[transformers] Both `max_new_tokens` (=80) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


[billing_error/empathetic]
  Sure, we apologize for any inconvenience this has caused! We've processed your request and issued a full refund of $<AMOUNT> for order #<ORDER_ID>. You should see the credit within 5-7 business days. If you have any further questions or concerns, please don't hesitate to reach out to us again. Thank


[transformers] Both `max_new_tokens` (=80) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[billing_error/formal]
  Sure, we apologize for any inconvenience this has caused! We've processed your request and issued a full refund of <AMOUNT> for order <ORDER_ID>. You should see the credit within 5-7 business days. If you have any further questions or concerns, please don't hesitate to reach out to us again. Thank y


[transformers] Both `max_new_tokens` (=80) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[delivery_delay/empathetic]
  Sure, here's an example of a support reply for the given customer message:

Thank you for reaching out! We apologize for any inconvenience caused by the delay in your order. Our records show that your order ( ORDER_ID) was shipped on DATE and should have arrived within 5 business days. Please allow 

[delivery_delay/formal]
  Sure, here's an example of a support reply for the given customer message:

Thank you for reaching out! We apologize for any inconvenience caused by the delay in your order. Our records show that your order ( ORDER_ID) was shipped on DATE and should have arrived within 5 business days. Please allow 

  Scenario             Style            SAF     TCI     HSI  Spurious
----------------------------------------------------------------------
  battery_issue        empathetic    0.0261  0.0223  0.0000  —
  battery_issue        formal       -0.0185 -0.0223  0.0000  —
  wrong_item           empathetic    0.0237  0.0206  0.0000  —
  wrong_i

In [27]:
# ─── Cell 17 — Verify saved .pkl files ───────────────────────────────────────

import os, pickle, torch
from pathlib import Path

cache_dir = Path('.style_cache')
pkl_files = sorted(cache_dir.glob('*.pkl'))

print('=' * 60)
print('Stored vectors in .style_cache/')
print('=' * 60)

if not pkl_files:
    print('  (no .pkl files found — did Cells 9 and 10 run successfully?)')
else:
    for pkl_path in pkl_files:
        size_kb = pkl_path.stat().st_size / 1024
        with open(pkl_path, 'rb') as f:
            v = pickle.load(f)
        print(f'  {pkl_path.name:<45} {size_kb:6.2f} KB  '
              f'shape={list(v.shape)}  norm={v.norm():.4f}')

print('=' * 60)
print(f'Cache dir (absolute): {cache_dir.resolve()}')
print()
print('Next steps:')
print('  1. Download the .style_cache/ folder')
print('  2. Place it next to main.py')
print(f'  3. Set in .env: LOCAL_MODEL_NAME={MODEL_NAME}')
print(f'         STEER_LAYER={cfg["layer"]}')
print(f'         STEER_ALPHA={cfg["alpha"]}')
print('         STYLE_CACHE_DIR=.style_cache')
print('  4. python main.py battery_issue')
print()
print('✓ Vector build pipeline complete!')

Stored vectors in .style_cache/
  style_vec_empathetic_pca.pkl                   20.39 KB  shape=[5120]  norm=1.0000
  style_vec_formal_pca.pkl                       20.39 KB  shape=[5120]  norm=1.0000
Cache dir (absolute): /home/pakdd/github repo/EMNLP 2026/A2A_KG_Llama-2-13b-chat-hf/.style_cache

Next steps:
  1. Download the .style_cache/ folder
  2. Place it next to main.py
  3. Set in .env: LOCAL_MODEL_NAME=meta-llama/Llama-2-13b-chat-hf
         STEER_LAYER=28
         STEER_ALPHA=12.0
         STYLE_CACHE_DIR=.style_cache
  4. python main.py battery_issue

✓ Vector build pipeline complete!
